# Task 2 — Correlation & Relationship Analysis

This notebook is **self-contained**: it does not depend on Task 1's kernel state. It reloads the cleaned dataset (or regenerates it if the file isn't present) before analyzing it.

## 1. Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import os

if os.path.exists('ecommerce_orders_clean.csv'):
    df = pd.read_csv('ecommerce_orders_clean.csv', parse_dates=['order_date'])
    print('Loaded cleaned dataset from Task 1 output.')
else:
    print('Clean file not found — regenerating from source (see Task 1 for full logic).')
    import pandas as pd
    import numpy as np

    # ---------------------------------------------------------------
    # Self-contained data generation
    # This block is identical across all three notebooks so each one
    # can run independently (no shared kernel state required).
    # Seeded for reproducibility.
    # ---------------------------------------------------------------
    np.random.seed(42)

    n = 1200
    categories = ['Electronics','Clothing','Home & Kitchen','Books','Beauty','Sports','Toys','Grocery']
    payment_methods = ['Credit Card','Debit Card','Cash on Delivery','Digital Wallet']
    regions = ['North','South','East','West','Central']
    statuses = ['Delivered','Cancelled','Returned','Pending']
    status_weights = [0.62, 0.16, 0.14, 0.08]

    start_date = pd.Timestamp('2023-01-01')
    end_date = pd.Timestamp('2025-06-30')
    date_range_days = (end_date - start_date).days

    df = pd.DataFrame({
        'order_id': [f'ORD{100000+i}' for i in range(n)],
        'customer_id': np.random.randint(1000, 1500, size=n),
        'order_date': [start_date + pd.Timedelta(days=int(x)) for x in np.random.randint(0, date_range_days, size=n)],
        'product_category': np.random.choice(categories, size=n),
        'quantity': np.random.randint(1, 6, size=n),
        'unit_price': np.round(np.random.gamma(3, 25, size=n), 2),
        'discount_percent': np.random.choice([0,5,10,15,20,25], size=n, p=[0.35,0.2,0.2,0.15,0.07,0.03]),
        'payment_method': np.random.choice(payment_methods, size=n),
        'shipping_cost': np.round(np.random.uniform(2,15,size=n),2),
        'delivery_days': np.random.randint(1,15,size=n),
        'customer_rating': np.random.choice([1,2,3,4,5], size=n, p=[0.05,0.08,0.17,0.35,0.35]),
        'order_status': np.random.choice(statuses, size=n, p=status_weights),
        'region': np.random.choice(regions, size=n),
    })

    df['total_amount'] = np.round(
        df['quantity'] * df['unit_price'] * (1 - df['discount_percent']/100) + df['shipping_cost'], 2
    )

    # Inject realistic data-quality issues so the pipeline has something genuine to catch
    missing_idx = np.random.choice(df.index, size=25, replace=False)
    df.loc[missing_idx, 'customer_rating'] = np.nan
    df = pd.concat([df, df.sample(8, random_state=1)], ignore_index=True)

    df = df.drop_duplicates().copy()
    df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].median())

print(f'Working dataset: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

## 2. Correlation Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

num_cols = ['quantity','unit_price','discount_percent','shipping_cost',
            'delivery_days','customer_rating','total_amount']
corr = df[num_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Matrix — Numeric Order Features')
plt.tight_layout()
plt.show()

In [ ]:
print('Correlation of each feature with total_amount:')
print(corr['total_amount'].sort_values(ascending=False))

## 3. Pairwise Relationships

In [ ]:
sns.pairplot(df[num_cols + []].sample(min(300, len(df)), random_state=0), diag_kind='kde', plot_kws={'alpha':0.5, 's':20})
plt.suptitle('Pairwise Relationships (300-row sample for readability)', y=1.02)
plt.show()

## 4. Outlier Detection (Quantified via IQR)

Outliers are not just plotted — they are counted using the standard 1.5×IQR rule, so the conclusion is a number, not an impression.

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(22, 5))
outlier_summary = {}
for i, col in enumerate(num_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(col)
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = df[(df[col] < low) | (df[col] > high)].shape[0]
    outlier_summary[col] = {'count': n_out, 'pct': round(n_out/len(df)*100, 2)}
plt.tight_layout()
plt.show()

outlier_df = pd.DataFrame(outlier_summary).T.sort_values('count', ascending=False)
print('Outlier counts (1.5x IQR rule):')
print(outlier_df)

## 5. Segment-Level Analysis

In [ ]:
revenue_by_category = df.groupby('product_category')['total_amount'].sum().sort_values(ascending=False)
plt.figure(figsize=(10,5))
sns.barplot(x=revenue_by_category.values, y=revenue_by_category.index, hue=revenue_by_category.index, palette='viridis', legend=False)
plt.title('Total Revenue by Product Category')
plt.xlabel('Total Revenue')
plt.tight_layout()
plt.show()
print(revenue_by_category)

In [ ]:
cancel_return = df[df['order_status'].isin(['Cancelled','Returned'])]
rate_by_payment = (cancel_return.groupby('payment_method').size() / df.groupby('payment_method').size() * 100).sort_values(ascending=False)
print('Cancellation + return rate by payment method (%):')
print(rate_by_payment.round(2))

## 6. Summary

- Correlation is reported as an explicit ranked table, not just a heatmap to eyeball.
- Outliers are counted per column with the IQR rule — exact counts and percentages, not 'a few outliers visible'.
- Revenue and cancellation/return rate are broken down by segment (category, payment method) to surface actionable differences rather than dataset-wide averages only.